# Merge Result-Property Descriptions into the Object-Types RST Reference

Pure text processing, no Toolkit calls needed - merges `result_property_descriptions.json` (from `result_props_descriptions.ipynb`) into `object_types_props_results_snippet_global_mapped.rst` (from `object_types.ipynb`), widening every "Result Properties" table from one column (Name) to three (Name, Description, Standard Physical Unit). Also adds an object category label (with the toolkit method to create the element) under every object type's heading, from `object_type_categories.json` (from `object_types_categories.ipynb`). The same content is also written as `object_type_reference.json` (copied to `src/sir3stoolkit/mantle/` manually, like the RST; queried with `SIR3S_Model_Dataframes.get_object_type_reference()`). Writes a new file here in `output/` - does not touch `docs/source/object_types_props_results_snippet_global_mapped.rst` itself; that gets copied over manually once reviewed.

## Load inputs

In [1]:
import re
import json
from pathlib import Path

output_dir = Path.cwd().resolve() / "output"
rst_path = output_dir / "object_types_props_results_snippet_global_mapped.rst"
desc_path = output_dir / "result_property_descriptions.json"

rst_text = rst_path.read_text(encoding="utf-8")
desc_map = json.loads(desc_path.read_text(encoding="utf-8"))
len(rst_text), len(desc_map)

(141643, 776)

## Widen every "Result Properties" table

Object-type headings in this (mapped) RST are the table codes (e.g. `ROHR`), matching the `OBJTYPE` half of `result_property_descriptions.json`'s `OBJTYPE~ATTRTYPE` keys directly - no further mapping needed. Sections with "No result properties found." are left untouched. Missing lookups (e.g. `MAINELEMENT`, a toolkit-reported result property with no corresponding `.MX1` channel in the source data) get an empty Description/Unit cell rather than being dropped, so the property itself is never lost from the reference. Description/Unit are wrapped in double backticks like the existing Name/Value-Type columns - also neutralizes stray `*`/`|` characters some units contain (e.g. `[KW|MW]`), which would otherwise be parsed as RST markup.

In [2]:
heading_pat = re.compile(r"^(?P<name>[^\r\n]+)\r?\n\^{3,}\r?\n", re.MULTILINE)
matches = list(heading_pat.finditer(rst_text))

result_block_pat = re.compile(
    r"(Result Properties\r?\n\"+\r?\n\r?\n)"
    r"(\.\. list-table::\r?\n\s*:header-rows: 1\r?\n\r?\n\s*\* - Name\r?\n)"
    r"((?:\s*\* - ``[^`]+``\r?\n)+)"
)
row_pat = re.compile(r"\* - ``([^`]+)``")

def build_new_header(header_block):
    return header_block.rstrip("\n") + "\n     - Description\n     - Standard Physical Unit\n"

def build_new_rows(rows_text, section_name):
    out = []
    for p in row_pat.findall(rows_text):
        entry = desc_map.get(f"{section_name}~{p}")
        # .strip(): a stray leading/trailing space right against the closing ``
        # breaks RST inline-literal parsing (seen once in the source MX1 TITLE data itself).
        title = (entry["TITLE"] or "").strip() if entry else ""
        unit = (entry["UNIT"] or "").strip() if entry else ""
        title_cell = f"``{title}``" if title else ""
        unit_cell = f"``{unit}``" if unit else ""
        out.append(f"   * - ``{p}``\n     - {title_cell}\n     - {unit_cell}\n")
    return "".join(out)

In [3]:
replacements = []
sections_matched = 0
rows_total = 0
rows_described = 0

for i, m in enumerate(matches):
    start = m.end()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(rst_text)
    section_name = m.group("name").strip()
    section_text = rst_text[start:end]

    rb = result_block_pat.search(section_text)
    if not rb:
        continue
    sections_matched += 1

    props = row_pat.findall(rb.group(3))
    rows_total += len(props)
    rows_described += sum(1 for p in props if f"{section_name}~{p}" in desc_map)

    new_block = rb.group(1) + build_new_header(rb.group(2)) + build_new_rows(rb.group(3), section_name)
    replacements.append((start + rb.start(), start + rb.end(), new_block))

print(f"{sections_matched} object-type sections widened, "
      f"{rows_described}/{rows_total} result-property rows matched a description ({rows_described / rows_total:.1%})")

59 object-type sections widened, 760/840 result-property rows matched a description (90.5%)


In [4]:
parts = []
cursor = 0
for start, end, new_block in replacements:
    parts.append(rst_text[cursor:start])
    parts.append(new_block)
    cursor = end
parts.append(rst_text[cursor:])
new_rst_text = "".join(parts)
len(new_rst_text)

184545

## Fix up the preamble note

The source note (inherited from `object_types.ipynb`'s raw output) still says "Result properties are listed by name only." - no longer true now that Description/Unit are added. Replaced with an accurate sentence, and a note that Description/Unit are in German (as embedded in SIR 3S's own `.MX1` output).

In [5]:
OLD_NOTE_TAIL = "Result properties are listed by name only."
NEW_NOTE_TAIL = (
    "Result properties additionally list a Description and Standard Physical Unit where known, "
    "sourced from SIR 3S's own .MX1 output plus a handful of manual additions (see "
    "result_props_descriptions.ipynb). Both are in German, as embedded in SIR 3S itself.\n"
    "   Use :meth:`sir3stoolkit.mantle.dataframes.SIR3S_Model_Dataframes.get_result_file_properties` "
    "to check what physical units were actually used in your local calculation, which can deviate "
    "from the standard physical units here."
)

assert OLD_NOTE_TAIL in new_rst_text, "Expected preamble sentence not found - has the source note changed?"
new_rst_text = new_rst_text.replace(OLD_NOTE_TAIL, NEW_NOTE_TAIL, 1)
print("Preamble note updated.")

Preamble note updated.


## Add object category labels

Adds an `Object Category: ...` line under every `Object Type: ``...``` line, from `object_type_categories.json`, linking the toolkit method(s) to create the element with where one exists. Tables and table rows refer to each other by table code, since that's what the section headings are. Fails if any object-type section has no category or any category has no section, so this step can't silently fall out of sync with `object_types.ipynb`.

In [6]:
categories = json.loads((output_dir / "object_type_categories.json").read_text(encoding="utf-8"))

CATEGORY_LABELS = {
    "node": "Node",
    "connecting_element": "Connecting element (two end nodes)",
    "four_end_node_element": "Connecting element with four end nodes",
    "bypass_element": "Bypass element (one end node)",
    "table": "Table",
    "table_row": "Table row",
    "signal_model_element": "Signal model element",
    "signal_model_connection": "Signal model connection line",
    "graphical_object": "Graphical object",
    "container_or_block": "Container or block element",
    "attached_element": "Element attached to another network element",
    "sub_record": "Sub-record of another element",
    "settings": "Model-wide settings",
    "reference_data": "Reference data (referenced by other elements)",
    "not_insertable": "Not insertable (no SIR 3S table behind this object type)",
    "unknown": "Unknown",
}

def meth(qualified_name):
    # shown with its class (SIR3S_Model/SIR3S_View), so it's clear which one to call it on
    return f":meth:`{qualified_name} <sir3stoolkit.core.wrapper.{qualified_name}>`"

def table_code(object_type_name):
    return f"``{categories[object_type_name]['table_name'] or object_type_name}``"

def build_category_line(object_type_name):
    entry = categories[object_type_name]
    label = CATEGORY_LABELS[entry["category"]]
    if "row_type" in entry:
        label += f" (rows: {table_code(entry['row_type'])})"
    if "table_type" in entry:
        label += f" of {table_code(entry['table_type'])}"
    if entry["create_with"]:
        label += " - create with " + ", ".join(meth(m) for m in entry["create_with"])
    if entry["category"] == "table":
        label += f", add rows with {meth('SIR3S_Model.AddTableRow')}"
    if entry["category"] == "four_end_node_element":
        label += f", read its end nodes with {meth('SIR3S_Model.GetEndNodes')}"
    return f"Object Category: {label}"

build_category_line("Pipe"), build_category_line("PipeTable")

('Object Category: Connecting element (two end nodes) - create with :meth:`SIR3S_Model.AddNewPipe <sir3stoolkit.core.wrapper.SIR3S_Model.AddNewPipe>`',
 'Object Category: Table (rows: ``DTRO_ROWD``) - create with :meth:`SIR3S_Model.InsertElement <sir3stoolkit.core.wrapper.SIR3S_Model.InsertElement>`, add rows with :meth:`SIR3S_Model.AddTableRow <sir3stoolkit.core.wrapper.SIR3S_Model.AddTableRow>`')

In [7]:
object_type_line_pat = re.compile(r"^Object Type: ``(?P<name>[^`]+)``\r?\n", re.MULTILINE)
labelled = []

def add_category_line(m):
    labelled.append(m.group("name"))
    return m.group(0) + "\n" + build_category_line(m.group("name")) + "\n"

new_rst_text = object_type_line_pat.sub(add_category_line, new_rst_text)

assert len(labelled) == len(set(labelled)), "An object type has more than one section"
assert set(labelled) == set(categories), (
    f"Sections without category: {sorted(set(labelled) - set(categories))}, "
    f"categories without section: {sorted(set(categories) - set(labelled))}"
)
print(f"{len(labelled)} object-type sections labelled")

149 object-type sections labelled


Also mentions the labels in the preamble note, right after the sentence added in the previous step.

In [8]:
NOTE_ANCHOR = "which can deviate from the standard physical units here."
CATEGORY_NOTE = (
    "\n   Each object type lists its Object Category - determined via the Toolkit API where possible, by hand "
    "otherwise, Unknown where neither was certain - and the toolkit method to create it, where one was shown to "
    "create a real instance of it (see object_types_categories.ipynb)."
)

assert new_rst_text.count(NOTE_ANCHOR) == 1, "Expected preamble sentence not found - has the note changed?"
new_rst_text = new_rst_text.replace(NOTE_ANCHOR, NOTE_ANCHOR + CATEGORY_NOTE, 1)
print("Preamble note extended.")

Preamble note extended.


## Save

In [9]:
out_path = output_dir / "object_types_props_results_snippet_global_mapped_with_descriptions.rst"
out_path.write_text(new_rst_text, encoding="utf-8")
print(f"Wrote {out_path}")

Wrote C:\Users\aUsername\3S\sir3stoolkit\docs\source\tutorials\SIR3S_Model_Mantle\TutorialTest_Assets\object_types\output\object_types_props_results_snippet_global_mapped_with_descriptions.rst


## Write the object type reference (JSON)

The same content as the RST, per `ObjectTypes` member name: `table_name` (`None` for the three types without one), `category`, `create_with`, `row_type`/`table_type` (tables and their rows), `insertable_in`, `properties` (name -> value type) and `result_properties` (name -> `description`/`unit`, `None` where unknown). Properties and result-property names are read back from the source RST's tables, so the JSON can't list anything the RST doesn't. JSON has no comments, so the RST's preamble note goes in as data: `{"notice": [...], "object_types": {...}}`, the notice taken from the RST itself (`:meth:` markup reduced to the plain name).

In [10]:
section_pat = re.compile(r"^Object Type: ``(?P<name>[^`]+)``\n(?P<body>.*?)(?=^[^\n]+\n\^{3,}\n|\Z)",
                         re.MULTILINE | re.DOTALL)
# value types can contain a backtick themselves (dictionary`2, nullable`1)
property_row_pat = re.compile(r"^   \* - ``([^`]+)``\n     - ``(.+)``$", re.MULTILINE)
result_row_pat = re.compile(r"^   \* - ``([^`]+)``", re.MULTILINE)

def description_and_unit(table_name, result_property):
    entry = desc_map.get(f"{table_name}~{result_property}") or {}
    return {"description": (entry.get("TITLE") or "").strip() or None,
            "unit": (entry.get("UNIT") or "").strip() or None}

reference = {}
for m in section_pat.finditer(rst_text):
    name = m.group("name")
    properties_text, _, results_text = m.group("body").partition("Result Properties\n")
    entry = {key: value for key, value in categories[name].items() if key != "source"}
    entry["properties"] = dict(property_row_pat.findall(properties_text))
    entry["result_properties"] = {p: description_and_unit(entry["table_name"], p)
                                  for p in result_row_pat.findall(results_text)}
    reference[name] = entry

assert set(reference) == set(categories), sorted(set(reference) ^ set(categories))
n_properties = sum(len(e["properties"]) for e in reference.values())
n_results = sum(len(e["result_properties"]) for e in reference.values())
assert n_results == rows_total, (n_results, rows_total)
print(f"{len(reference)} object types, {n_properties} properties, {n_results} result properties")

149 object types, 2588 properties, 840 result properties


In [11]:
note_match = re.search(r"^\.\. note:: (?P<first>.*)\n(?P<rest>(?:   .*\n)*)", new_rst_text, re.MULTILINE)
notice = [note_match.group("first")] + [line.strip() for line in note_match.group("rest").splitlines()]
notice = [re.sub(r":meth:`([^`]+)`", r"\1", line) for line in notice]

reference_path = output_dir / "object_type_reference.json"
reference_json = {"notice": notice, "object_types": dict(sorted(reference.items()))}
reference_path.write_text(json.dumps(reference_json, indent=1, ensure_ascii=False) + "\n",
                          encoding="utf-8")
print(f"Wrote {reference_path}")

Wrote C:\Users\aUsername\3S\sir3stoolkit\docs\source\tutorials\SIR3S_Model_Mantle\TutorialTest_Assets\object_types\output\object_type_reference.json
